# Optimización, Dimensionamiento y FinOps de Datos

## Objetivos

- Calcular dimensionamiento de **storage, compute y network** según 3 V
- Aplicar técnicas de **optimización** (particionamiento, clustering, compresión)
- Implementar estrategias de **FinOps** para control de costos
- Comparar modelos de costo en **AWS, Azure, GCP**
- Diseñar **observabilidad** con métricas y alertas

## 1. Dimensionamiento Detallado según 3 V

### Metodología de Dimensionamiento

Para dimensionar correctamente una arquitectura de datos:

1. **Analizar los 3 V:**
   - 📊 **Volumen:** Datos actuales + crecimiento proyectado
   - ⚡ **Velocidad:** Tasa de ingesta (eventos/seg) + latencia requerida
   - 🎨 **Variedad:** Formatos, esquemas, fuentes heterogéneas

2. **Dimensionar recursos:**
   - Storage: Volumen × retención × compresión × réplicas
   - Compute: Throughput requerido / capacidad por nodo
   - Network: Ancho de banda pico + overhead (30-50%)

3. **Validar con SLOs:**
   - Latencia objetivo (P95, P99)
   - Disponibilidad (uptime %)
   - Error rate aceptable

In [ ]:
# Calculadora avanzada de dimensionamiento
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

class DimensionamientoCalculator:
    def __init__(self, nombre_proyecto):
        self.nombre = nombre_proyecto
        self.resultados = {}
    
    def calcular_storage(self, volumen_diario_gb, dias_retencion_hot, 
                         dias_retencion_cold, factor_compresion=0.3, replicas=3):
        """Calcula storage necesario en hot y cold tiers"""
        # Hot storage (acceso frecuente)
        vol_hot_raw = volumen_diario_gb * dias_retencion_hot
        vol_hot_comprimido = vol_hot_raw * factor_compresion
        vol_hot_con_replicas = vol_hot_comprimido * replicas
        
        # Cold storage (archivo)
        vol_cold_raw = volumen_diario_gb * dias_retencion_cold
        vol_cold_comprimido = vol_cold_raw * factor_compresion
        vol_cold_con_replicas = vol_cold_comprimido * replicas
        
        self.resultados['storage'] = {
            'hot_tb': vol_hot_con_replicas / 1024,
            'cold_tb': vol_cold_con_replicas / 1024,
            'total_tb': (vol_hot_con_replicas + vol_cold_con_replicas) / 1024
        }
        return self.resultados['storage']
    
    def calcular_compute(self, eventos_por_segundo_pico, bytes_por_evento, 
                        cpu_por_1k_eventos=0.5, ram_por_gb_cache=8):
        """Calcula recursos de compute necesarios"""
        # Throughput en GB/hora
        gb_por_hora = (eventos_por_segundo_pico * bytes_por_evento * 3600) / (1024**3)
        
        # CPUs necesarias (basado en eventos/segundo)
        vcpus_necesarias = (eventos_por_segundo_pico / 1000) * cpu_por_1k_eventos
        vcpus_recomendadas = np.ceil(vcpus_necesarias * 1.5)  # 50% overhead
        
        # RAM necesaria (cache + overhead)
        ram_cache_gb = gb_por_hora * 2  # 2 horas de cache
        ram_total_gb = np.ceil(ram_cache_gb + ram_por_gb_cache)
        
        self.resultados['compute'] = {
            'vcpus': int(vcpus_recomendadas),
            'ram_gb': int(ram_total_gb),
            'throughput_gb_hora': round(gb_por_hora, 2)
        }
        return self.resultados['compute']
    
    def calcular_network(self, eventos_por_segundo_pico, bytes_por_evento):
        """Calcula ancho de banda necesario"""
        # Bits por segundo
        bps = eventos_por_segundo_pico * bytes_por_evento * 8
        mbps = bps / (1024 * 1024)
        gbps = mbps / 1024
        
        # Overhead de 40% (headers, retransmisiones, etc)
        mbps_con_overhead = mbps * 1.4
        
        self.resultados['network'] = {
            'mbps_base': round(mbps, 2),
            'mbps_recomendado': round(mbps_con_overhead, 2),
            'gbps': round(gbps, 3)
        }
        return self.resultados['network']
    
    def generar_reporte(self):
        """Genera reporte completo de dimensionamiento"""
        print(f"{'='*60}")
        print(f"REPORTE DE DIMENSIONAMIENTO: {self.nombre}")
        print(f"{'='*60}")
        
        if 'storage' in self.resultados:
            s = self.resultados['storage']
            print(f"\n📦 STORAGE:")
            print(f"  Hot (Standard):  {s['hot_tb']:.2f} TB")
            print(f"  Cold (Archive):  {s['cold_tb']:.2f} TB")
            print(f"  TOTAL:          {s['total_tb']:.2f} TB")
        
        if 'compute' in self.resultados:
            c = self.resultados['compute']
            print(f"\n🖥️  COMPUTE:")
            print(f"  vCPUs:          {c['vcpus']}")
            print(f"  RAM:            {c['ram_gb']} GB")
            print(f"  Throughput:     {c['throughput_gb_hora']} GB/hora")
        
        if 'network' in self.resultados:
            n = self.resultados['network']
            print(f"\n🌐 NETWORK:")
            print(f"  Base:           {n['mbps_base']} Mbps")
            print(f"  Recomendado:    {n['mbps_recomendado']} Mbps (con overhead)")
            if n['gbps'] >= 1:
                print(f"  Equivalente:    {n['gbps']} Gbps")
        
        print(f"\n{'='*60}")

# Ejemplo de uso
calc = DimensionamientoCalculator("E-commerce Analytics Platform")

# Parámetros del caso
calc.calcular_storage(
    volumen_diario_gb=800,
    dias_retencion_hot=90,
    dias_retencion_cold=730,  # 2 años adicionales
    factor_compresion=0.25,
    replicas=3
)

calc.calcular_compute(
    eventos_por_segundo_pico=15000,
    bytes_por_evento=2048,  # 2 KB
    cpu_por_1k_eventos=0.6,
    ram_por_gb_cache=16
)

calc.calcular_network(
    eventos_por_segundo_pico=15000,
    bytes_por_evento=2048
)

calc.generar_reporte()

## 2. Técnicas de Optimización

### A. Particionamiento

**Objetivo:** Reducir datos escaneados mediante pruning de particiones.

**Estrategias comunes:**
- **Temporal:** Por fecha/hora (más común)
- **Geográfico:** Por región/país
- **Por categoría:** Por tipo de producto, canal, etc.
- **Hash:** Distribuir uniformemente para paralelismo

**Trade-offs:**
| Beneficio | Costo |
|-----------|-------|
| Queries filtradas 10-100x más rápidas | Overhead en escritura (más archivos) |
| Menor costo de procesamiento | Demasiadas particiones (small files) pueden degradar |

### B. Clustering / Z-Ordering

**Objetivo:** Co-localizar datos relacionados físicamente.

- **Snowflake:** Clustering keys (automático)
- **BigQuery:** Clustering (hasta 4 columnas)
- **Delta Lake:** Z-ORDER (múltiples columnas)

### C. Compresión y Formatos

| Formato | Compresión Típica | Lectura | Escritura | Uso |
|---------|-------------------|---------|-----------|-----|
| **CSV** | Gzip: 3-5x | Lenta | Rápida | Ingesta inicial |
| **Parquet** | Snappy: 3-4x | Muy rápida | Media | Analytics (recomendado) |
| **ORC** | Zlib: 4-6x | Muy rápida | Media | Hive/Spark |
| **Avro** | Snappy: 2-3x | Media | Rápida | Streaming |

### D. Materialized Views y Caching

- **Materialized Views:** Pre-computar agregaciones costosas
- **Result Cache:** BigQuery, Snowflake cachean queries idénticas (24h)
- **BI Engine:** Acelerar dashboards con cache en memoria

In [ ]:
# Demostración de impacto de particionamiento
import time

# Simular dataset grande
np.random.seed(42)
n_registros = 1_000_000

df_sin_particionar = pd.DataFrame({
    'fecha': pd.date_range('2024-01-01', periods=n_registros, freq='30s'),
    'cliente_id': np.random.randint(1, 10000, n_registros),
    'monto': np.random.uniform(10, 1000, n_registros),
    'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste'], n_registros)
})

# Query sin particionamiento: escanear todo el dataset
start = time.time()
resultado_sin_part = df_sin_particionar[
    (df_sin_particionar['fecha'] >= '2024-06-01') & 
    (df_sin_particionar['fecha'] < '2024-06-02')
]
tiempo_sin_part = time.time() - start

# Simular particionamiento (agrupar por mes)
df_sin_particionar['mes'] = df_sin_particionar['fecha'].dt.to_period('M')
particiones = dict(tuple(df_sin_particionar.groupby('mes')))

# Query con particionamiento: solo leer partición necesaria
start = time.time()
mes_objetivo = pd.Period('2024-06', freq='M')
if mes_objetivo in particiones:
    df_particion = particiones[mes_objetivo]
    resultado_con_part = df_particion[
        (df_particion['fecha'] >= '2024-06-01') & 
        (df_particion['fecha'] < '2024-06-02')
    ]
tiempo_con_part = time.time() - start

# Comparación
print("=== IMPACTO DEL PARTICIONAMIENTO ===")
print(f"Dataset total: {len(df_sin_particionar):,} registros")
print(f"Registros filtrados: {len(resultado_sin_part):,}")
print(f"\nTiempo SIN particionamiento: {tiempo_sin_part*1000:.2f} ms")
print(f"Tiempo CON particionamiento: {tiempo_con_part*1000:.2f} ms")
print(f"\n🚀 Speedup: {tiempo_sin_part/tiempo_con_part:.1f}x más rápido")
print(f"💰 Datos escaneados: {100*len(particiones[mes_objetivo])/len(df_sin_particionar):.1f}% del total")

## 3. FinOps: Gestión de Costos en la Nube

### ¿Qué es FinOps?

**FinOps** (Financial Operations) es la práctica de gestionar costos en la nube con:
- **Visibilidad:** ¿Cuánto gastamos y en qué?
- **Optimización:** ¿Cómo reducir sin afectar rendimiento?
- **Accountability:** ¿Quién es responsable de cada costo?

### Pilares de FinOps para Datos

#### 1. Tagging/Labeling
Etiquetar recursos para atribuir costos:
```yaml
tags:
  environment: production
  team: data-analytics
  project: customer-360
  cost-center: CC-1234
```

#### 2. Presupuestos y Alertas
- Definir presupuesto mensual por dominio/proyecto
- Alertas al 50%, 80%, 100% del presupuesto
- Acciones automáticas (suspender clusters, notificar)

#### 3. Right-Sizing
- Analizar uso real vs capacidad provisionada
- Reducir tamaños de clusters infrautilizados
- Auto-scaling para cargas variables

#### 4. Chargeback vs Showback
- **Showback:** Reportar costos a cada equipo (informativo)
- **Chargeback:** Facturar internamente a cada equipo (accountability real)

#### 5. Storage Tiering
- **Hot:** Datos activos (Standard storage)
- **Warm:** Acceso ocasional (Infrequent Access)
- **Cold:** Archivo histórico (Glacier/Archive)

## 4. Modelos de Costo por Plataforma Cloud

### AWS

**Storage (S3):**
- Standard: $0.023/GB/mes
- Infrequent Access: $0.0125/GB/mes
- Glacier: $0.004/GB/mes

**Compute (EMR, Redshift):**
- EMR: $costo_EC2 + $0.27/hora/nodo
- Redshift: $0.25-$13.04/hora según tamaño

**Egress:**
- Primeros 100 GB/mes: gratis
- 100 GB - 10 TB: $0.09/GB
- Mismo región: gratis

### Azure

**Storage (Blob):**
- Hot: $0.0184/GB/mes
- Cool: $0.01/GB/mes
- Archive: $0.002/GB/mes

**Compute (Synapse, Databricks):**
- Synapse: $1.20-$122/hora según DWU
- Databricks: $costo_VM + $0.15-$0.55 DBU

### GCP

**Storage (Cloud Storage):**
- Standard: $0.020/GB/mes
- Nearline: $0.010/GB/mes
- Coldline: $0.004/GB/mes

**Compute (BigQuery):**
- On-demand: $5/TB procesado
- Slots (reserva): $2,000/mes por 100 slots
- Storage: $0.020/GB/mes activo, $0.010 long-term

**Nota:** Precios aproximados (región US), consultar pricing calculators oficiales.

In [ ]:
# Simulador de costos multi-cloud
class CostSimulator:
    def __init__(self):
        # Precios aproximados (USD)
        self.precios = {
            'aws': {
                'storage_standard_gb_mes': 0.023,
                'storage_glacier_gb_mes': 0.004,
                'compute_hora': 2.50,  # m5.2xlarge promedio
                'egress_gb': 0.09
            },
            'azure': {
                'storage_hot_gb_mes': 0.0184,
                'storage_archive_gb_mes': 0.002,
                'compute_hora': 2.80,  # D8s v3 promedio
                'egress_gb': 0.087
            },
            'gcp': {
                'storage_standard_gb_mes': 0.020,
                'storage_coldline_gb_mes': 0.004,
                'bigquery_tb_procesado': 5.00,
                'egress_gb': 0.12
            }
        }
    
    def calcular_costo_mensual(self, cloud, storage_hot_tb, storage_cold_tb, 
                                compute_horas, egress_tb, tb_procesado=0):
        """Calcula costo mensual para una plataforma"""
        precios = self.precios[cloud]
        
        if cloud == 'aws':
            costo_storage = (storage_hot_tb * 1024 * precios['storage_standard_gb_mes'] + 
                            storage_cold_tb * 1024 * precios['storage_glacier_gb_mes'])
            costo_compute = compute_horas * precios['compute_hora']
            costo_egress = egress_tb * 1024 * precios['egress_gb']
            
        elif cloud == 'azure':
            costo_storage = (storage_hot_tb * 1024 * precios['storage_hot_gb_mes'] + 
                            storage_cold_tb * 1024 * precios['storage_archive_gb_mes'])
            costo_compute = compute_horas * precios['compute_hora']
            costo_egress = egress_tb * 1024 * precios['egress_gb']
            
        elif cloud == 'gcp':
            costo_storage = (storage_hot_tb * 1024 * precios['storage_standard_gb_mes'] + 
                            storage_cold_tb * 1024 * precios['storage_coldline_gb_mes'])
            costo_compute = tb_procesado * precios['bigquery_tb_procesado']  # BigQuery on-demand
            costo_egress = egress_tb * 1024 * precios['egress_gb']
        
        return {
            'storage': costo_storage,
            'compute': costo_compute,
            'egress': costo_egress,
            'total': costo_storage + costo_compute + costo_egress
        }
    
    def comparar_clouds(self, storage_hot_tb, storage_cold_tb, compute_horas, 
                       egress_tb, tb_procesado):
        """Compara costos entre las 3 nubes"""
        resultados = {}
        
        for cloud in ['aws', 'azure', 'gcp']:
            resultados[cloud] = self.calcular_costo_mensual(
                cloud, storage_hot_tb, storage_cold_tb, compute_horas, 
                egress_tb, tb_procesado
            )
        
        # Mostrar comparación
        print("=" * 70)
        print("COMPARACIÓN DE COSTOS MENSUALES (USD)")
        print("=" * 70)
        print(f"{'Cloud':<10} {'Storage':<15} {'Compute':<15} {'Egress':<15} {'TOTAL':<15}")
        print("-" * 70)
        
        for cloud, costos in resultados.items():
            print(f"{cloud.upper():<10} ${costos['storage']:<14,.2f} ${costos['compute']:<14,.2f} "
                  f"${costos['egress']:<14,.2f} ${costos['total']:<14,.2f}")
        
        # Identificar más económico
        mas_barato = min(resultados.items(), key=lambda x: x[1]['total'])
        print("\n" + "=" * 70)
        print(f"💰 Más económico: {mas_barato[0].upper()} (${mas_barato[1]['total']:,.2f}/mes)")
        
        return resultados

# Ejemplo de uso
sim = CostSimulator()

# Escenario: arquitectura mediana
costos = sim.comparar_clouds(
    storage_hot_tb=20,        # 20 TB en hot storage
    storage_cold_tb=100,      # 100 TB en cold/archive
    compute_horas=500,        # 500 horas/mes de compute (AWS/Azure)
    egress_tb=5,              # 5 TB transferidos fuera
    tb_procesado=150          # 150 TB procesados (GCP BigQuery)
)

In [ ]:
# Implementación de Chargeback: Atribuir costos a equipos
# Simulación de costos por proyecto/equipo

datos_uso = {
    'proyecto': ['CustomerAnalytics', 'FraudDetection', 'Marketing', 'SupplyChain'],
    'equipo': ['Data Science', 'Security', 'Marketing', 'Operations'],
    'storage_tb': [15, 8, 12, 25],
    'compute_horas': [120, 200, 80, 150],
    'egress_gb': [500, 200, 1500, 300]
}

df_uso = pd.DataFrame(datos_uso)

# Tarifas internas (pueden ser diferentes a precios cloud reales)
TARIFA_STORAGE_TB = 25.00  # $/TB/mes
TARIFA_COMPUTE_HORA = 3.00  # $/hora
TARIFA_EGRESS_GB = 0.10    # $/GB

# Calcular costos por proyecto
df_uso['costo_storage'] = df_uso['storage_tb'] * TARIFA_STORAGE_TB
df_uso['costo_compute'] = df_uso['compute_horas'] * TARIFA_COMPUTE_HORA
df_uso['costo_egress'] = df_uso['egress_gb'] * TARIFA_EGRESS_GB
df_uso['costo_total'] = df_uso['costo_storage'] + df_uso['costo_compute'] + df_uso['costo_egress']

print("=" * 80)
print("REPORTE DE CHARGEBACK - OCTUBRE 2025")
print("=" * 80)
print(df_uso[['proyecto', 'equipo', 'costo_storage', 'costo_compute', 
              'costo_egress', 'costo_total']].to_string(index=False))

print("\n" + "=" * 80)
print(f"TOTAL ORGANIZACIÓN: ${df_uso['costo_total'].sum():,.2f}")

# Visualizar distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Costo por proyecto
axes[0].barh(df_uso['proyecto'], df_uso['costo_total'], color='steelblue')
axes[0].set_xlabel('Costo Total (USD)')
axes[0].set_title('Costo por Proyecto')
axes[0].grid(axis='x', alpha=0.3)

# Gráfico 2: Desglose por tipo de costo
componentes = df_uso[['costo_storage', 'costo_compute', 'costo_egress']].sum()
axes[1].pie(componentes, labels=['Storage', 'Compute', 'Egress'], 
            autopct='%1.1f%%', startangle=90, colors=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Distribución de Costos')

plt.tight_layout()
plt.show()

# Identificar proyecto más costoso
proyecto_max = df_uso.loc[df_uso['costo_total'].idxmax()]
print(f"\n⚠️  Proyecto con mayor costo: {proyecto_max['proyecto']} (${proyecto_max['costo_total']:,.2f})")
print(f"    Equipo responsable: {proyecto_max['equipo']}")

## 5. Observabilidad: Métricas y Alertas

### Métricas Clave para Arquitectura de Datos

#### A. Métricas de Rendimiento
| Métrica | Objetivo | Herramienta |
|---------|----------|-------------|
| **Throughput** | > 1 GB/s | CloudWatch, Datadog |
| **Latencia (P95)** | < 5 segundos | Prometheus, Grafana |
| **Query Time (P99)** | < 10 segundos | Snowflake Query History |
| **CPU Utilization** | 60-80% (óptimo) | Azure Monitor |
| **Memory Utilization** | < 85% | GCP Monitoring |

#### B. Métricas de Calidad
- **Frescura:** Tiempo desde ingesta hasta disponibilidad
- **Completitud:** % registros sin nulos en campos críticos
- **Error Rate:** % pipelines fallidos

#### C. Métricas de Costo
- **Costo por TB procesado**
- **Costo por query**
- **Tendencia mensual** (detectar spikes)

### Estrategia de Alertas

**Niveles de severidad:**
1. **Critical:** Sistema caído, requiere acción inmediata (PagerDuty)
2. **Warning:** SLO en riesgo, investigar en horas de trabajo
3. **Info:** Notificación de cambios, sin acción requerida

**Ejemplos:**
```yaml
alertas:
  - nombre: "Latencia Alta"
    condicion: "P95_latency > 10s durante 5 min"
    severidad: warning
    canal: slack-data-ops
  
  - nombre: "Costo Diario Excedido"
    condicion: "costo_diario > presupuesto * 1.5"
    severidad: critical
    canal: email-finops
  
  - nombre: "Pipeline Fallido"
    condicion: "pipeline_status == FAILED"
    severidad: critical
    canal: pagerduty
```

In [ ]:
# Simulador de dashboard de observabilidad
import datetime

# Generar métricas sintéticas para 30 días
np.random.seed(123)
dias = pd.date_range('2025-10-01', '2025-10-30', freq='D')

metricas_diarias = pd.DataFrame({
    'fecha': dias,
    'throughput_gbps': np.random.uniform(0.8, 1.5, len(dias)),
    'latencia_p95_seg': np.random.gamma(2, 2, len(dias)),
    'queries_ejecutadas': np.random.randint(5000, 15000, len(dias)),
    'error_rate_pct': np.random.uniform(0.1, 2.5, len(dias)),
    'costo_usd': np.random.uniform(1500, 3500, len(dias))
})

# Inyectar algunos eventos anómalos
metricas_diarias.loc[10, 'latencia_p95_seg'] = 15  # Spike de latencia
metricas_diarias.loc[10, 'error_rate_pct'] = 8    # Spike de errores
metricas_diarias.loc[22, 'costo_usd'] = 6500      # Spike de costo

# Dashboard de métricas
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Latencia (con SLO)
axes[0, 0].plot(metricas_diarias['fecha'], metricas_diarias['latencia_p95_seg'], 
                marker='o', color='steelblue', label='P95 Latency')
axes[0, 0].axhline(y=5, color='green', linestyle='--', label='SLO: 5s')
axes[0, 0].axhline(y=10, color='red', linestyle='--', label='Critical: 10s')
axes[0, 0].set_ylabel('Latencia P95 (segundos)')
axes[0, 0].set_title('Latencia de Queries (P95)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Error Rate
axes[0, 1].plot(metricas_diarias['fecha'], metricas_diarias['error_rate_pct'], 
                marker='s', color='coral', label='Error Rate')
axes[0, 1].axhline(y=1, color='orange', linestyle='--', label='Warning: 1%')
axes[0, 1].axhline(y=5, color='red', linestyle='--', label='Critical: 5%')
axes[0, 1].set_ylabel('Error Rate (%)')
axes[0, 1].set_title('Tasa de Errores')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].tick_params(axis='x', rotation=45)

# 3. Costo Diario
axes[1, 0].bar(metricas_diarias['fecha'], metricas_diarias['costo_usd'], 
               color='seagreen', alpha=0.7, label='Costo Diario')
costo_promedio = metricas_diarias['costo_usd'].mean()
axes[1, 0].axhline(y=costo_promedio, color='blue', linestyle='--', 
                   label=f'Promedio: ${costo_promedio:.0f}')
axes[1, 0].axhline(y=costo_promedio * 1.5, color='red', linestyle='--', 
                   label='Umbral: +50%')
axes[1, 0].set_ylabel('Costo (USD)')
axes[1, 0].set_title('Costo Diario')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Queries Ejecutadas
axes[1, 1].plot(metricas_diarias['fecha'], metricas_diarias['queries_ejecutadas'], 
                marker='^', color='purple', label='Queries/Día')
axes[1, 1].set_ylabel('Queries Ejecutadas')
axes[1, 1].set_title('Volumen de Queries')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.suptitle('Dashboard de Observabilidad - Octubre 2025', 
             fontsize=16, y=1.02, fontweight='bold')
plt.show()

# Resumen de alertas
print("=" * 60)
print("ALERTAS GENERADAS EN EL PERÍODO")
print("=" * 60)

alertas = []
for idx, row in metricas_diarias.iterrows():
    if row['latencia_p95_seg'] > 10:
        alertas.append(f"[CRITICAL] {row['fecha'].date()}: Latencia {row['latencia_p95_seg']:.1f}s > 10s")
    if row['error_rate_pct'] > 5:
        alertas.append(f"[CRITICAL] {row['fecha'].date()}: Error rate {row['error_rate_pct']:.1f}% > 5%")
    if row['costo_usd'] > costo_promedio * 1.5:
        alertas.append(f"[WARNING] {row['fecha'].date()}: Costo ${row['costo_usd']:.0f} excede presupuesto")

if alertas:
    for alerta in alertas:
        print(alerta)
else:
    print("✅ No hay alertas en el período")

print(f"\n📊 Total queries ejecutadas: {metricas_diarias['queries_ejecutadas'].sum():,}")
print(f"💰 Costo total del mes: ${metricas_diarias['costo_usd'].sum():,.2f}")
print(f"⚡ Throughput promedio: {metricas_diarias['throughput_gbps'].mean():.2f} GB/s")

## 6. Ejercicio Integrador

**Escenario:** Plataforma de streaming de video

**Requisitos:**
- **Volumen:** 5 TB/día de logs + metadatos
- **Velocidad:** 50,000 eventos/segundo en pico
- **Variedad:** Logs JSON, metadatos Parquet, imágenes (thumbnails)
- **Retención:** 30 días hot, 1 año cold
- **SLO:** Latencia P95 < 3s para queries de analytics
- **Presupuesto:** $15,000/mes

**Tareas:**

1. **Dimensionamiento:**
   - Calcular storage (hot + cold) con compresión 3.5x
   - Calcular ancho de banda necesario
   - Determinar vCPUs y RAM para procesamiento

2. **Optimización:**
   - Proponer estrategia de particionamiento (temporal/geográfico)
   - Seleccionar formato óptimo (Parquet/Avro/ORC) y justificar
   - Identificar candidatos para materialized views

3. **FinOps:**
   - Comparar costo en AWS vs GCP vs Azure
   - Diseñar estrategia de storage tiering
   - Definir 3 acciones para reducir 20% del costo

4. **Observabilidad:**
   - Listar 5 métricas clave a monitorear
   - Definir 3 alertas críticas con umbrales
   - Proponer dashboard para equipo de operaciones

In [ ]:
# Plantilla de solución
solucion = '''
=== SOLUCIÓN EJERCICIO INTEGRADOR ===

1. DIMENSIONAMIENTO:
   A. Storage:
      - Hot (30 días): [calcular TB]
      - Cold (1 año): [calcular TB]
      - Total con compresión 3.5x: [resultado]
   
   B. Network:
      - Eventos/seg pico: 50,000
      - Bytes por evento: [estimar]
      - Mbps necesario: [calcular]
   
   C. Compute:
      - vCPUs: [calcular]
      - RAM: [calcular GB]

2. OPTIMIZACIÓN:
   A. Particionamiento:
      [Proponer estrategia: fecha_hora + region]
   
   B. Formato:
      [Seleccionar Parquet/Avro/ORC y justificar]
   
   C. Materialized Views:
      [Identificar agregaciones frecuentes]

3. FINOPS:
   A. Comparación de costos:
      AWS:   $[calcular]
      Azure: $[calcular]
      GCP:   $[calcular]
   
   B. Storage tiering:
      [Estrategia de migración hot→cold]
   
   C. 3 Acciones de reducción de costo:
      1) [acción 1]
      2) [acción 2]
      3) [acción 3]

4. OBSERVABILIDAD:
   A. 5 Métricas clave:
      1) [métrica 1]
      2) [métrica 2]
      3) [métrica 3]
      4) [métrica 4]
      5) [métrica 5]
   
   B. 3 Alertas críticas:
      1) [alerta + umbral]
      2) [alerta + umbral]
      3) [alerta + umbral]
   
   C. Dashboard:
      [Describir paneles principales]
'''

print(solucion)

## 7. Resumen y Mejores Prácticas

### ✅ Dimensionamiento
- Basar decisiones en los **3 V** (Volumen, Velocidad, Variedad)
- Considerar **crecimiento proyectado** (no solo estado actual)
- Provisionar con **overhead** (30-50%) para picos
- Revisar dimensionamiento **trimestralmente**

### ✅ Optimización
- **Particionar** por columnas frecuentemente filtradas
- Usar **formatos columnares** (Parquet) con compresión (Snappy/Zstd)
- **Clustering/Z-ORDER** en columnas de JOIN
- **Evitar SELECT *** y proyectar solo columnas necesarias
- **Materialized views** para agregaciones repetitivas

### ✅ FinOps
- **Tagging obligatorio** de todos los recursos
- **Presupuestos** por equipo/proyecto con alertas
- **Chargeback/Showback** mensual con desglose
- **Right-sizing** continuo basado en métricas reales
- **Auto-suspend** de recursos fuera de horario
- **Storage tiering** automático (lifecycle policies)

### ✅ Observabilidad
- Monitorear **métricas de rendimiento** (latencia, throughput, error rate)
- Monitorear **métricas de costo** (diario/semanal/mensual)
- Monitorear **métricas de calidad** (frescura, completitud)
- Definir **SLOs realistas** alineados a negocio
- Configurar **alertas por niveles** (info/warning/critical)
- **Dashboards para diferentes audiencias** (ops, finops, ejecutivos)

### 📚 Herramientas Recomendadas
- **Observabilidad:** Datadog, Grafana, Prometheus, CloudWatch
- **FinOps:** Cloudability, CloudHealth, Kubecost, native cloud cost tools
- **Optimización:** dbt, Great Expectations, Soda, Monte Carlo
- **Calculadoras:** AWS Pricing Calculator, Azure Pricing Calculator, GCP Pricing Calculator

### 🎯 Próximo Nivel (Senior)
- FinOps avanzado: cost anomaly detection, capacity planning
- Optimización a escala: query profiling, index strategies
- Observabilidad predictiva: ML para forecasting de métricas
- Multi-cloud cost optimization y data transfer strategies